In [ ]:
!pip install mcap mcap-ros2-support

from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import os

#mcap_path = "./data/dataset_path"
#mcap_path = "./data/dataset_path"
mcap_path = "./data/dataset_path"

required_topics = [
    "/estimated_robot_state", "odom", "joint_state",
    "/unitree/contact_state", "/unitree/imu", "/unitree/raw_contact"
]

if not os.path.exists(mcap_path):
    print("File does not exist.")
else:
    with open(mcap_path, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])
        
        # Get all topics in the file
        summary = reader.get_summary()
        topics_found = {ch.topic for ch in summary.channels.values()}
        print("Topics found:", topics_found)
        
        for topic in required_topics:
            if topic in topics_found:
                print(f"  ✓ {topic} found")
            else:
                print(f"  ✗ {topic} NOT found")
                

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=["/unitree/imu"]
    ):
        print(f"Topic: {channel.topic}")
        print(f"Message: {ros_msg}")
        break  # Remove to read all messages

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import os

#mcap_path = "./data/dataset_path"
mcap_path = "./data/dataset_path"
#mcap_path = "./data/dataset_path"


with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    print("=" * 60)
    print("SCHEMA INFO (message field definitions)")
    print("=" * 60)
    summary = reader.get_summary()
    for schema_id, schema in summary.schemas.items():
        print(f"\nSchema: {schema.name}")
        if schema.data:
            print(schema.data.decode("utf-8", errors="ignore"))

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    print("\n" + "=" * 60)
    print("SAMPLE MESSAGES FROM /lowstate")
    print("=" * 60)
    count = 0
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/lowstate"]):
        print(f"\n--- Message {count+1} ---")
        print(ros_msg)
        count += 1
        if count >= 3:
            break

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    print("\n" + "=" * 60)
    print("SAMPLE MESSAGES FROM /sportmodestate")
    print("=" * 60)
    count = 0
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=["/sportmodestate"]):
        print(f"\n--- Message {count+1} ---")
        print(ros_msg)
        count += 1
        if count >= 3:
            break

In [ ]:
# THIS HERE SHOULD EXTRACT THE CONTACT STATE DATA FROM /lowstate AND SAVE TO CSV FOR ANALYSIS
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import pandas as pd
import matplotlib.pyplot as plt
import os

#mcap_path = "./data/dataset_path"
mcap_path = "./data/dataset_path"
#mcap_path = "./data/dataset_path"

# ── Extract contact-related data from /lowstate ──────────────────────────────
rows = []

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=["/lowstate"]
    ):
        ms = ros_msg.motor_state

        row = {
            "tick": ros_msg.tick,

            # ── Raw foot force sensors (equivalent to /unitree/raw_contact) ──
            "foot_force_FR": ros_msg.foot_force[0],
            "foot_force_FL": ros_msg.foot_force[1],
            "foot_force_RR": ros_msg.foot_force[2],
            "foot_force_RL": ros_msg.foot_force[3],

            # ── Estimated foot force ──────────────────────────────────────────
            "foot_force_est_FR": ros_msg.foot_force_est[0],
            "foot_force_est_FL": ros_msg.foot_force_est[1],
            "foot_force_est_RR": ros_msg.foot_force_est[2],
            "foot_force_est_RL": ros_msg.foot_force_est[3],

            # ── Binary contact state (True if force above threshold) ──────────
            "contact_FR": int(ros_msg.foot_force[0] > 20),
            "contact_FL": int(ros_msg.foot_force[1] > 20),
            "contact_RR": int(ros_msg.foot_force[2] > 20),
            "contact_RL": int(ros_msg.foot_force[3] > 20),

            # ── Joint torques per leg (useful for contact estimation) ─────────
            "tau_FR_calf": ms[2].tau_est,
            "tau_FL_calf": ms[5].tau_est,
            "tau_RR_calf": ms[8].tau_est,
            "tau_RL_calf": ms[11].tau_est,
        }
        rows.append(row)

df = pd.DataFrame(rows)

# ── Normalize tick to start at 0 ─────────────────────────────────────────────
df["tick"] = df["tick"] - df["tick"].iloc[0]

print("Columns:", df.columns.tolist())
print("\nStats per column:")
print(df.describe())
print("\nFirst few rows:")
print(df.head())

# ── Save CSV ──────────────────────────────────────────────────────────────────
output_dir = "./data/dataset_path"
os.makedirs(output_dir, exist_ok=True)
base_name = os.path.splitext(os.path.basename(mcap_path))[0]
csv_path = os.path.join(output_dir, f"{base_name}_contact.csv")
df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

# ── Plot raw foot forces ──────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Raw foot forces
ax = axes[0]
for leg, color in zip(["FR", "FL", "RR", "RL"], ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    ax.plot(df["tick"], df[f"foot_force_{leg}"], label=leg, color=color)
ax.set_ylabel("Raw Foot Force")
ax.set_title("Raw Foot Forces (Go2 /lowstate)")
ax.legend()
ax.grid(True)

# Binary contact state
ax = axes[1]
for i, (leg, color) in enumerate(zip(["FR", "FL", "RR", "RL"], ["tab:blue", "tab:orange", "tab:green", "tab:red"])):
    ax.plot(df["tick"], df[f"contact_{leg}"] + i * 1.2, label=leg, color=color)
ax.set_ylabel("Contact (0/1, shifted)")
ax.set_title("Binary Contact State (threshold > 20)")
ax.legend()
ax.grid(True)

# Calf torques
ax = axes[2]
for leg, color in zip(["FR", "FL", "RR", "RL"], ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    ax.plot(df["tick"], df[f"tau_{leg}_calf"], label=leg, color=color)
ax.set_ylabel("Calf Torque [Nm]")
ax.set_title("Calf Joint Torques (contact proxy)")
ax.set_xlabel("Tick")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_contact.png"), dpi=150)
plt.show()
print("Plot saved.")

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import pandas as pd
import numpy as np
import os

#mcap_path = "./data/dataset_path"
mcap_path = "./data/dataset_path"
output_dir = "./data/dataset_path"
os.makedirs(output_dir, exist_ok=True)

# ── Go2 Kinematics (body frame, meters) ──────────────────────────────────────
# Hip offsets from body center
HIP_OFFSETS = {
    "FR": np.array([ 0.1934,  -0.0465, 0.0]),
    "FL": np.array([ 0.1934,   0.0465, 0.0]),
    "RR": np.array([-0.1934,  -0.0465, 0.0]),
    "RL": np.array([-0.1934,   0.0465, 0.0]),
}
L_THIGH = 0.213   # thigh link length [m]
L_CALF  = 0.213   # calf link length [m]
L_HIP   = 0.0955  # hip lateral offset [m]
SIDE    = {"FR": -1, "FL": 1, "RR": -1, "RL": 1}  # right=-1, left=+1

def forward_kinematics(hip_q, thigh_q, calf_q, leg):
    """Returns EE position in body frame for one leg."""
    s = SIDE[leg]
    offset = HIP_OFFSETS[leg]

    # Hip rotation (abduction/adduction)
    hip_lateral = s * L_HIP
    p_hip = np.array([
        0.0,
        hip_lateral * np.cos(hip_q),
        hip_lateral * np.sin(hip_q)
    ])

    # Thigh rotation
    p_thigh = np.array([
        0.0,
        -L_THIGH * np.sin(thigh_q),
        -L_THIGH * np.cos(thigh_q)
    ])

    # Calf rotation (relative to thigh)
    p_calf = np.array([
        0.0,
        -L_CALF * np.sin(thigh_q + calf_q),
        -L_CALF * np.cos(thigh_q + calf_q)
    ])

    ee_pos = offset + p_hip + p_thigh + p_calf
    return ee_pos

# ── Joint index map (motor_state indices) ────────────────────────────────────
JOINTS = {
    "FR": (0, 1, 2),
    "FL": (3, 4, 5),
    "RR": (6, 7, 8),
    "RL": (9, 10, 11),
}

rows = []

with open(mcap_path, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    prev_ee = {leg: None for leg in ["FR", "FL", "RR", "RL"]}
    prev_tick = None

    for schema, channel, message, ros_msg in reader.iter_decoded_messages(
        topics=["/lowstate"]
    ):
        ms   = ros_msg.motor_state
        imu  = ros_msg.imu_state
        tick = ros_msg.tick

        row = {
            "tick": tick,

            # ── IMU - body frame ──────────────────────────────────────────────
            # Linear acceleration (body frame) [m/s²]
            "acc_x":   imu.accelerometer[0],
            "acc_y":   imu.accelerometer[1],
            "acc_z":   imu.accelerometer[2],
            # Angular velocity (body frame) [rad/s]
            "gyro_x":  imu.gyroscope[0],
            "gyro_y":  imu.gyroscope[1],
            "gyro_z":  imu.gyroscope[2],
            # Orientation
            "roll":    imu.rpy[0],
            "pitch":   imu.rpy[1],
            "yaw":     imu.rpy[2],
            "quat_x":  imu.quaternion[0],
            "quat_y":  imu.quaternion[1],
            "quat_z":  imu.quaternion[2],
            "quat_w":  imu.quaternion[3],

            # ── Contact state ─────────────────────────────────────────────────
            "contact_FR": int(ros_msg.foot_force[0] > 20),
            "contact_FL": int(ros_msg.foot_force[1] > 20),
            "contact_RR": int(ros_msg.foot_force[2] > 20),
            "contact_RL": int(ros_msg.foot_force[3] > 20),
            "foot_force_FR": ros_msg.foot_force[0],
            "foot_force_FL": ros_msg.foot_force[1],
            "foot_force_RR": ros_msg.foot_force[2],
            "foot_force_RL": ros_msg.foot_force[3],
        }

        # ── Joint positions & velocities ──────────────────────────────────────
        for leg, (hi, ti, ci) in JOINTS.items():
            row[f"q_{leg}_hip"]   = ms[hi].q
            row[f"q_{leg}_thigh"] = ms[ti].q
            row[f"q_{leg}_calf"]  = ms[ci].q
            row[f"dq_{leg}_hip"]  = ms[hi].dq
            row[f"dq_{leg}_thigh"]= ms[ti].dq
            row[f"dq_{leg}_calf"] = ms[ci].dq
            row[f"tau_{leg}_hip"]   = ms[hi].tau_est
            row[f"tau_{leg}_thigh"] = ms[ti].tau_est
            row[f"tau_{leg}_calf"]  = ms[ci].tau_est

        # ── EE position via Forward Kinematics (body frame) ───────────────────
        dt = (tick - prev_tick) / 1000.0 if prev_tick is not None else None

        for leg, (hi, ti, ci) in JOINTS.items():
            ee_pos = forward_kinematics(
                ms[hi].q, ms[ti].q, ms[ci].q, leg
            )
            row[f"ee_pos_{leg}_x"] = ee_pos[0]
            row[f"ee_pos_{leg}_y"] = ee_pos[1]
            row[f"ee_pos_{leg}_z"] = ee_pos[2]

            # EE velocity: finite difference of FK positions
            if prev_ee[leg] is not None and dt and dt > 0:
                ee_vel = (ee_pos - prev_ee[leg]) / dt
            else:
                ee_vel = np.array([0.0, 0.0, 0.0])

            row[f"ee_vel_{leg}_x"] = ee_vel[0]
            row[f"ee_vel_{leg}_y"] = ee_vel[1]
            row[f"ee_vel_{leg}_z"] = ee_vel[2]

            prev_ee[leg] = ee_pos

        prev_tick = tick
        rows.append(row)

# ── Save ──────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
df["tick"] = df["tick"] - df["tick"].iloc[0]

base_name = os.path.splitext(os.path.basename(mcap_path))[0]
out_path = os.path.join(output_dir, f"{base_name}_full.csv")
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}  ({len(df)} rows, {len(df.columns)} columns)")
print("\nColumns:", df.columns.tolist())
print("\nEE position sample (FR leg, body frame):")
print(df[["tick", "ee_pos_FR_x", "ee_pos_FR_y", "ee_pos_FR_z"]].head(5))
print("\nEE velocity sample (FR leg):")
print(df[["tick", "ee_vel_FR_x", "ee_vel_FR_y", "ee_vel_FR_z"]].head(5))
print("\nIMU sample:")
print(df[["tick", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]].head(5))

In [ ]:
# ── Cell 3: Signal plot per leg ───────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import os

# ── Path resolution (widget or fallback) ─────────────────────────────────────
#FALLBACK_PATH = "./data/dataset_path"
FALLBACK_PATH = "./data/dataset_path"
mcap_path = bag_paths_selected[0] if bag_paths_selected else FALLBACK_PATH
base_name = os.path.splitext(os.path.basename(mcap_path))[0]
output_dir = "./data/dataset_path"

# Load from the CSV saved in Cell 2
df = pd.read_csv(os.path.join(output_dir, f"{base_name}_full.csv"))

LEGS   = ["FR", "FL", "RR", "RL"]
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
CONTACT_THRESHOLD = 20

print(f"Loaded: {base_name}_full.csv  ({len(df)} rows)")
print(f"Tick range: {df['tick'].iloc[0]} → {df['tick'].iloc[-1]}")

# ── Plot 1: all 4 raw contact signals overlaid ────────────────────────────────
plt.figure(figsize=(14, 5))
for leg, color in zip(LEGS, COLORS):
    plt.plot(df["tick"], df[f"foot_force_{leg}"], label=leg, color=color, lw=0.8)
plt.axhline(CONTACT_THRESHOLD, color="k", lw=0.8, ls="--", label=f"threshold ({CONTACT_THRESHOLD})")
plt.xlabel("Tick")
plt.ylabel("Raw contact sensor reading (int16)")
plt.title(f"Raw contact signals per leg — {base_name}")
plt.legend(title="Leg")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_contact_overlay.png"), dpi=150)
plt.show()

# ── Plot 2: 4 subplots — raw contact per leg ─────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f"Raw contact signal per leg — {base_name}", fontsize=12, fontweight="bold")

for i, (leg, color) in enumerate(zip(LEGS, COLORS)):
    ax = axes[i]
    ax.plot(df["tick"], df[f"foot_force_{leg}"], color=color, lw=0.8, label="raw")
    ax.fill_between(df["tick"],
                    df[f"contact_{leg}"] * df[f"foot_force_{leg}"].max(),
                    alpha=0.15, color=color, label="contact region")
    ax.axhline(CONTACT_THRESHOLD, color="k", lw=0.6, ls="--", alpha=0.6)
    ax.set_ylabel("Force (int16)")
    ax.set_title(f"{leg} — raw foot_force  (contact HIGH = shaded)")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Tick")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_contact_per_leg.png"), dpi=150)
plt.show()

# ── Plot 3: 4 subplots — foot Z position per leg (from FK) ───────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f"Foot Z position per leg (FK, body frame) — {base_name}", fontsize=12, fontweight="bold")

for i, (leg, color) in enumerate(zip(LEGS, COLORS)):
    ax = axes[i]
    ax.plot(df["tick"], df[f"ee_pos_{leg}_z"], color=color, lw=0.8, label="foot Z")
    ax.fill_between(df["tick"],
                    df[f"ee_pos_{leg}_z"].min(),
                    df[f"ee_pos_{leg}_z"].max(),
                    where=df[f"contact_{leg}"] == 1,
                    alpha=0.15, color=color, label="contact=1")
    ax.set_ylabel("Z (m)")
    ax.set_title(f"{leg} — foot Z  (shaded = contact sensor HIGH)")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Tick")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_foot_z_per_leg.png"), dpi=150)
plt.show()

# ── Plot 4: combined — raw contact + foot Z twin axes per leg ─────────────────
# This is the key defect diagnostic plot
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle(f"Contact sensor vs foot Z — defect diagnostic — {base_name}",
            fontsize=12, fontweight="bold")

for i, (leg, color) in enumerate(zip(LEGS, COLORS)):
    ax = axes[i]
    ax2 = ax.twinx()

    ax.plot(df["tick"], df[f"foot_force_{leg}"], color=color, lw=0.8, label="raw force")
    ax.axhline(CONTACT_THRESHOLD, color="k", lw=0.6, ls="--", alpha=0.5)
    ax.set_ylabel("Foot force (int16)", color=color)
    ax.tick_params(axis="y", labelcolor=color)

    ax2.plot(df["tick"], df[f"ee_pos_{leg}_z"], color="gray", lw=0.8,
            alpha=0.7, label="foot Z")
    ax2.set_ylabel("Foot Z (m)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    ax.set_title(f"{leg} — force (colour) vs foot Z (gray)  |  contact should peak when Z is lowest")
    ax.grid(True, alpha=0.3)

    # combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")

axes[-1].set_xlabel("Tick")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_contact_vs_z_diagnostic.png"), dpi=150)
plt.show()

# ── Quick stats ───────────────────────────────────────────────────────────────
print(f"\nTick range : {df['tick'].iloc[0]} → {df['tick'].iloc[-1]}")
print(f"Rows       : {len(df)}")

print("\nRaw foot force stats:")
print(df[[f"foot_force_{l}" for l in LEGS]].describe().round(2))

print("\nFoot Z stats (expect ~−0.27 to −0.35 m when standing):")
print(df[[f"ee_pos_{l}_z" for l in LEGS]].describe().round(4))

print(f"\nContact ratio per leg (fraction of time contact=1, threshold={CONTACT_THRESHOLD}):")
for leg in LEGS:
    ratio = df[f"contact_{leg}"].mean()
    bar = "█" * int(ratio * 40)
    print(f"  {leg}: {ratio:.2%}  {bar}")

In [ ]:
# ── Cell 4: Contact sensor delay analysis ────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import find_peaks
import os

#FALLBACK_PATH = "./data/dataset_path"
FALLBACK_PATH = "./data/dataset_path"
mcap_path  = bag_paths_selected[0] if bag_paths_selected else FALLBACK_PATH
base_name  = os.path.splitext(os.path.basename(mcap_path))[0]
output_dir = "./data/dataset_path"

df = pd.read_csv(os.path.join(output_dir, f"{base_name}_full.csv"))

LEGS              = ["FR", "FL", "RR", "RL"]
COLORS            = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
CONTACT_THRESHOLD = 20
SEARCH_WINDOW     = 50   # ticks to search around each Z minimum for the force edge

# ── Step 1: detect rising edges of contact signal ─────────────────────────────
# A rising edge = contact goes from 0 → 1
def get_rising_edges(contact_series):
    contact = contact_series.values
    edges = np.where((contact[:-1] == 0) & (contact[1:] == 1))[0] + 1
    return edges

# ── Step 2: find Z local minima (foot touchdown = Z at its lowest) ────────────
def get_z_minima(z_series, prominence=0.005):
    z = z_series.values
    # invert Z so minima become peaks for find_peaks
    peaks, props = find_peaks(-z, prominence=prominence, distance=20)
    return peaks

# ── Compute delay per leg ─────────────────────────────────────────────────────
delay_results = {}

for leg in LEGS:
    force_col   = f"foot_force_{leg}"
    contact_col = f"contact_{leg}"
    z_col       = f"ee_pos_{leg}_z"

    rising_edges = get_rising_edges(df[contact_col])
    z_minima     = get_z_minima(df[z_col])
    ticks        = df["tick"].values

    delays = []
    matched_edges   = []
    matched_minima  = []

    for edge_idx in rising_edges:
        edge_tick = ticks[edge_idx]
        # Find the closest Z minimum within ±SEARCH_WINDOW ticks
        candidates = z_minima[np.abs(z_minima - edge_idx) <= SEARCH_WINDOW]
        if len(candidates) == 0:
            continue
        closest_min_idx = candidates[np.argmin(np.abs(candidates - edge_idx))]
        min_tick  = ticks[closest_min_idx]
        delay     = int(edge_tick) - int(min_tick)   # positive = sensor lags Z
        delays.append(delay)
        matched_edges.append(edge_idx)
        matched_minima.append(closest_min_idx)

    delay_results[leg] = {
        "delays":          np.array(delays),
        "matched_edges":   matched_edges,
        "matched_minima":  matched_minima,
        "rising_edges":    rising_edges,
        "z_minima":        z_minima,
    }

# ── Print delay summary ───────────────────────────────────────────────────────
print("=" * 55)
print("CONTACT SENSOR DELAY ANALYSIS")
print(f"Search window: ±{SEARCH_WINDOW} ticks around each Z minimum")
print("Positive delay = sensor fires AFTER foot touches ground")
print("Negative delay = sensor fires BEFORE Z minimum (early)")
print("=" * 55)

for leg in LEGS:
    d = delay_results[leg]["delays"]
    if len(d) == 0:
        print(f"\n{leg}: no matched events found — check threshold or window")
        continue
    print(f"\n{leg}  (n={len(d)} touchdown events)")
    print(f"  mean delay : {d.mean():.1f} ticks")
    print(f"  std        : {d.std():.1f} ticks")
    print(f"  median     : {np.median(d):.1f} ticks")
    print(f"  min / max  : {d.min()} / {d.max()} ticks")
    if abs(d.mean()) < 5:
        verdict = "✓ sensor appears well-synchronised"
    elif d.mean() > 15:
        verdict = "⚠ sensor is consistently LAGGING — possible defect or filter delay"
    elif d.mean() < -15:
        verdict = "⚠ sensor fires early — check threshold, may be too low"
    else:
        verdict = "~ small delay, within normal range"
    print(f"  verdict    : {verdict}")

# ── Plot: delay per event for each leg ───────────────────────────────────────
fig = plt.figure(figsize=(16, 14))
fig.suptitle(f"Contact sensor delay analysis — {base_name}", fontsize=13, fontweight="bold")
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.55, wspace=0.35)

for i, (leg, color) in enumerate(zip(LEGS, COLORS)):
    res    = delay_results[leg]
    delays = res["delays"]
    ticks  = df["tick"].values

    if len(delays) == 0:
        continue

    # Left: signal overlay with matched events annotated
    ax_sig = fig.add_subplot(gs[i, 0])
    ax_z   = ax_sig.twinx()

    ax_sig.plot(df["tick"], df[f"foot_force_{leg}"],
                color=color, lw=0.7, alpha=0.8, label="raw force")
    ax_sig.axhline(CONTACT_THRESHOLD, color="k", lw=0.6, ls="--", alpha=0.5)

    ax_z.plot(df["tick"], df[f"ee_pos_{leg}_z"],
              color="gray", lw=0.7, alpha=0.6, label="foot Z")

    # Mark rising edges (sensor fires)
    for ei in res["matched_edges"]:
        ax_sig.axvline(ticks[ei], color=color, lw=0.8, alpha=0.6)

    # Mark Z minima (true touchdown)
    for mi in res["matched_minima"]:
        ax_z.axvline(ticks[mi], color="gray", lw=0.8, ls="--", alpha=0.6)

    ax_sig.set_title(f"{leg}  |  colour=sensor edge  gray=Z min  "
                     f"mean delay={delays.mean():.1f} tk", fontsize=8)
    ax_sig.set_xlabel("Tick", fontsize=7)
    ax_sig.set_ylabel("Force", fontsize=7, color=color)
    ax_z.set_ylabel("Z (m)", fontsize=7, color="gray")
    ax_sig.tick_params(labelsize=7)
    ax_z.tick_params(labelsize=7)
    ax_sig.grid(True, lw=0.3, alpha=0.4)

    # Right: delay histogram
    ax_hist = fig.add_subplot(gs[i, 1])
    ax_hist.hist(delays, bins=20, color=color, alpha=0.75, edgecolor="white")
    ax_hist.axvline(0,            color="k",   lw=1.2, ls="-",  label="zero delay")
    ax_hist.axvline(delays.mean(), color="red", lw=1.2, ls="--", label=f"mean={delays.mean():.1f}")
    ax_hist.set_title(f"{leg} delay distribution", fontsize=8)
    ax_hist.set_xlabel("Delay (ticks)  +ve = sensor lags", fontsize=7)
    ax_hist.set_ylabel("Count", fontsize=7)
    ax_hist.legend(fontsize=7)
    ax_hist.tick_params(labelsize=7)
    ax_hist.grid(True, lw=0.3, alpha=0.4)

plt.savefig(os.path.join(output_dir, f"{base_name}_delay_analysis.png"), dpi=150,
            bbox_inches="tight")
plt.show()
print(f"\nPlot saved → {base_name}_delay_analysis.png")

In [ ]:
# ── Cell 5: Per-leg threshold + sensor bias investigation ────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

#FALLBACK_PATH = "./data/dataset_path"
FALLBACK_PATH = "./data/dataset_path"
mcap_path  = bag_paths_selected[0] if bag_paths_selected else FALLBACK_PATH
base_name  = os.path.splitext(os.path.basename(mcap_path))[0]
output_dir = "./data/dataset_path"

df = pd.read_csv(os.path.join(output_dir, f"{base_name}_full.csv"))

LEGS   = ["FR", "FL", "RR", "RL"]
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

# ── 1. Force value distribution when foot is clearly in swing (Z > -0.26 m) ──
# If force is high during swing → sensor is biased / never resets
print("=" * 60)
print("FORCE VALUES DURING SWING PHASE (foot Z > -0.26 m)")
print("Healthy sensor: values should be LOW (near baseline) here")
print("=" * 60)

for leg, color in zip(LEGS, COLORS):
    swing_mask = df[f"ee_pos_{leg}_z"] > -0.26
    swing_force = df.loc[swing_mask, f"foot_force_{leg}"]
    contact_mask = df[f"ee_pos_{leg}_z"] <= -0.26
    stance_force = df.loc[contact_mask, f"foot_force_{leg}"]

    print(f"\n{leg}")
    print(f"  swing  samples={len(swing_force):5d}  "
          f"mean={swing_force.mean():6.1f}  "
          f"std={swing_force.std():5.1f}  "
          f"max={swing_force.max():5.0f}  "
          f"% above threshold-20: {(swing_force > 20).mean()*100:.1f}%")
    print(f"  stance samples={len(stance_force):5d}  "
          f"mean={stance_force.mean():6.1f}  "
          f"std={stance_force.std():5.1f}  "
          f"max={stance_force.max():5.0f}  "
          f"% above threshold-20: {(stance_force > 20).mean()*100:.1f}%")

# ── 2. Find optimal threshold per leg ─────────────────────────────────────────
# Sweep thresholds and find where swing false-positive rate drops below 5%
print("\n" + "=" * 60)
print("OPTIMAL THRESHOLD SEARCH (target: <5% false positives in swing)")
print("=" * 60)

thresholds = np.arange(10, 100, 2)
optimal_thresholds = {}

for leg in LEGS:
    swing_mask  = df[f"ee_pos_{leg}_z"] > -0.26
    swing_force = df.loc[swing_mask, f"foot_force_{leg}"]
    for thresh in thresholds:
        fp_rate = (swing_force > thresh).mean()
        if fp_rate < 0.05:
            optimal_thresholds[leg] = int(thresh)
            print(f"  {leg}: optimal threshold = {thresh}  "
                  f"(swing false-positive rate = {fp_rate*100:.1f}%)")
            break
    else:
        optimal_thresholds[leg] = 20
        print(f"  {leg}: no clean threshold found — sensor may be stuck/biased")

# ── 3. Recompute contact with optimal thresholds ──────────────────────────────
print("\n" + "=" * 60)
print("CONTACT RATIO WITH OPTIMAL THRESHOLDS vs FIXED-20")
print("=" * 60)

for leg in LEGS:
    old_thresh = 20
    new_thresh = optimal_thresholds[leg]
    old_ratio  = (df[f"foot_force_{leg}"] > old_thresh).mean()
    new_ratio  = (df[f"foot_force_{leg}"] > new_thresh).mean()
    print(f"  {leg}:  thresh={old_thresh} → {new_ratio:.0%} contact  |  "
          f"thresh={new_thresh} → {new_ratio:.0%} contact")

# ── 4. Plot force distributions: swing vs stance per leg ─────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle(f"Force distribution — swing vs stance — {base_name}",
             fontsize=12, fontweight="bold")

for i, (leg, color) in enumerate(zip(LEGS, COLORS)):
    swing_mask  = df[f"ee_pos_{leg}_z"] > -0.26
    stance_mask = ~swing_mask
    swing_force  = df.loc[swing_mask,  f"foot_force_{leg}"]
    stance_force = df.loc[stance_mask, f"foot_force_{leg}"]

    # Top row: histogram
    ax = axes[0, i]
    ax.hist(swing_force,  bins=40, alpha=0.6, color="gray",  label="swing",  density=True)
    ax.hist(stance_force, bins=40, alpha=0.6, color=color,   label="stance", density=True)
    ax.axvline(20, color="k",   lw=1.0, ls="--", label="thresh=20")
    ax.axvline(optimal_thresholds[leg], color="red", lw=1.0,
               ls="--", label=f"optimal={optimal_thresholds[leg]}")
    ax.set_title(f"{leg} force distribution", fontsize=9)
    ax.set_xlabel("Foot force (int16)", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, lw=0.3, alpha=0.4)

    # Bottom row: force over time coloured by swing/stance
    ax2 = axes[1, i]
    ax2.plot(df["tick"], df[f"foot_force_{leg}"],
             color="lightgray", lw=0.5, zorder=1)
    ax2.scatter(df.loc[swing_mask,  "tick"],
                df.loc[swing_mask,  f"foot_force_{leg}"],
                c="gray", s=0.3, label="swing", zorder=2)
    ax2.scatter(df.loc[stance_mask, "tick"],
                df.loc[stance_mask, f"foot_force_{leg}"],
                c=color,  s=0.3, label="stance", zorder=3)
    ax2.axhline(20, color="k",   lw=0.8, ls="--", alpha=0.7)
    ax2.axhline(optimal_thresholds[leg], color="red", lw=0.8,
                ls="--", alpha=0.7)
    ax2.set_title(f"{leg} force vs tick  (gray=swing  colour=stance)", fontsize=9)
    ax2.set_xlabel("Tick", fontsize=8)
    ax2.set_ylabel("Force (int16)", fontsize=8)
    ax2.grid(True, lw=0.3, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{base_name}_threshold_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ── 5. Summary verdict ────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SENSOR HEALTH SUMMARY")
print("=" * 60)
for leg in LEGS:
    swing_mask  = df[f"ee_pos_{leg}_z"] > -0.26
    swing_force = df.loc[swing_mask, f"foot_force_{leg}"]
    fp_rate     = (swing_force > 20).mean()
    opt         = optimal_thresholds[leg]

    if fp_rate < 0.05:
        status = "✓ HEALTHY"
    elif fp_rate < 0.20:
        status = f"~ MARGINAL — use threshold={opt} instead of 20"
    else:
        status = f"✗ SUSPECT — {fp_rate*100:.0f}% false positives during swing"

    print(f"  {leg}: {status}  (swing FP rate={fp_rate*100:.1f}%,  "
          f"optimal threshold={opt})")

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import numpy as np
import pandas as pd
import os

# ============================================================
# ------------------- KINEMATICS (Go2) -----------------------
# ============================================================
HIP_OFFSETS = {
    "FR": np.array([ 0.1934, -0.0465, 0.0]),
    "FL": np.array([ 0.1934,  0.0465, 0.0]),
    "RR": np.array([-0.1934, -0.0465, 0.0]),
    "RL": np.array([-0.1934,  0.0465, 0.0]),
}
L_THIGH = 0.213
L_CALF  = 0.213
L_HIP   = 0.0955
SIDE    = {"FR": -1, "FL": 1, "RR": -1, "RL": 1}
JOINTS  = {"FR": (0,1,2), "FL": (3,4,5), "RR": (6,7,8), "RL": (9,10,11)}
LEGS    = ["FR", "FL", "RR", "RL"]

# deep-contact-estimator order: [RH,RF,LH,LF] = [RR,FR,RL,FL]
# Go2 flat: FR(0-2), FL(3-5), RR(6-8), RL(9-11)
JOINT_REORDER = [6, 7, 8, 0, 1, 2, 9, 10, 11, 3, 4, 5]

def forward_kinematics(hip_q, thigh_q, calf_q, leg):
    s      = SIDE[leg]
    offset = HIP_OFFSETS[leg]
    p_hip   = np.array([0.0,  s*L_HIP*np.cos(hip_q),           s*L_HIP*np.sin(hip_q)])
    p_thigh = np.array([0.0, -L_THIGH*np.sin(thigh_q),        -L_THIGH*np.cos(thigh_q)])
    p_calf  = np.array([0.0, -L_CALF*np.sin(thigh_q+calf_q),  -L_CALF*np.cos(thigh_q+calf_q)])
    return offset + p_hip + p_thigh + p_calf

# ============================================================
# ------------------- CONFIG ---------------------------------
# ============================================================
#FALLBACK_PATH  = "./data/dataset_path"
FALLBACK_PATH = "./data/dataset_path"
bag_paths      = bag_paths_selected if bag_paths_selected else [FALLBACK_PATH]
output_csv_dir = "./data/dataset_path"
output_npy_dir = "./data/dataset_path"

# Updated from Cell 5 optimal threshold analysis
CONTACT_THRESH_PER_LEG = {"FR": 16, "FL": 18, "RR": 18, "RL": 14}

os.makedirs(output_csv_dir, exist_ok=True)
os.makedirs(output_npy_dir, exist_ok=True)

# ============================================================
# ------------------- HELPERS --------------------------------
# ============================================================
        
def check_frame(name, array):
    if array.size == 0:
        print(f"  {name}: EMPTY")
        return
    max_val = np.nanmax(np.abs(array))
    # Different limits for different signal types
    if "vel" in name.lower() or "v " in name.lower():
        limit = 15.0   # EE velocity: fast swing can reach 6–10 m/s
    elif "acc" in name.lower():
        limit = 50.0   # IMU: gravity + dynamics, peaks ~30 m/s²
    else:
        limit = 5.0    # positions, joint angles
    status = "✓ looks correct" if max_val < limit else "⚠ check this"
    print(f"  {name}: {status} (max abs = {max_val:.3f}, limit={limit})")

# ============================================================
# ------------------- MAIN LOOP ------------------------------
# ============================================================
for bag_path in bag_paths:
    if not os.path.exists(bag_path):
        print(f"✗ Not found: {bag_path}")
        continue

    base_name = os.path.splitext(os.path.basename(bag_path))[0]
    print(f"\n{'='*60}\nProcessing: {base_name}\n{'='*60}")

    rows       = []
    prev_ee    = {leg: None for leg in LEGS}
    prev_tick  = None
    tick_diffs = []

    with open(bag_path, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])
        for schema, channel, message, ros_msg in reader.iter_decoded_messages(
            topics=["/lowstate"]
        ):
            ms   = ros_msg.motor_state
            imu  = ros_msg.imu_state
            tick = ros_msg.tick

            if prev_tick is not None:
                tick_diffs.append(tick - prev_tick)
            dt = (tick - prev_tick) / 1000.0 if prev_tick is not None else None

            row = {
                "tick":   tick,
                "time_s": tick / 1000.0,
                # IMU
                "acc_x":  imu.accelerometer[0], "acc_y": imu.accelerometer[1], "acc_z": imu.accelerometer[2],
                "gyro_x": imu.gyroscope[0],     "gyro_y": imu.gyroscope[1],    "gyro_z": imu.gyroscope[2],
                "roll":   imu.rpy[0], "pitch": imu.rpy[1], "yaw": imu.rpy[2],
                "quat_x": imu.quaternion[0], "quat_y": imu.quaternion[1],
                "quat_z": imu.quaternion[2], "quat_w": imu.quaternion[3],
                # Raw + estimated foot force
                "foot_force_FR": ros_msg.foot_force[0],
                "foot_force_FL": ros_msg.foot_force[1],
                "foot_force_RR": ros_msg.foot_force[2],
                "foot_force_RL": ros_msg.foot_force[3],
                "foot_force_est_FR": ros_msg.foot_force_est[0],
                "foot_force_est_FL": ros_msg.foot_force_est[1],
                "foot_force_est_RR": ros_msg.foot_force_est[2],
                "foot_force_est_RL": ros_msg.foot_force_est[3],
                # Binary contact with per-leg threshold
                "contact_FR": int(ros_msg.foot_force[0] > CONTACT_THRESH_PER_LEG["FR"]),
                "contact_FL": int(ros_msg.foot_force[1] > CONTACT_THRESH_PER_LEG["FL"]),
                "contact_RR": int(ros_msg.foot_force[2] > CONTACT_THRESH_PER_LEG["RR"]),
                "contact_RL": int(ros_msg.foot_force[3] > CONTACT_THRESH_PER_LEG["RL"]),
            }

            for leg, (hi, ti, ci) in JOINTS.items():
                row[f"q_{leg}_hip"]     = ms[hi].q
                row[f"q_{leg}_thigh"]   = ms[ti].q
                row[f"q_{leg}_calf"]    = ms[ci].q
                row[f"dq_{leg}_hip"]    = ms[hi].dq
                row[f"dq_{leg}_thigh"]  = ms[ti].dq
                row[f"dq_{leg}_calf"]   = ms[ci].dq
                row[f"tau_{leg}_hip"]   = ms[hi].tau_est
                row[f"tau_{leg}_thigh"] = ms[ti].tau_est
                row[f"tau_{leg}_calf"]  = ms[ci].tau_est

                ee_pos = forward_kinematics(ms[hi].q, ms[ti].q, ms[ci].q, leg)
                row[f"ee_pos_{leg}_x"] = ee_pos[0]
                row[f"ee_pos_{leg}_y"] = ee_pos[1]
                row[f"ee_pos_{leg}_z"] = ee_pos[2]

                ee_vel = (ee_pos - prev_ee[leg]) / dt if (prev_ee[leg] is not None and dt and dt > 0) else np.zeros(3)
                row[f"ee_vel_{leg}_x"] = ee_vel[0]
                row[f"ee_vel_{leg}_y"] = ee_vel[1]
                row[f"ee_vel_{leg}_z"] = ee_vel[2]
                prev_ee[leg] = ee_pos

            prev_tick = tick
            rows.append(row)

    df = pd.DataFrame(rows)
    df["time_s"] = df["time_s"] - df["time_s"].iloc[0]
    df["tick"]   = df["tick"]   - df["tick"].iloc[0]

    # ── Hz check ─────────────────────────────────────────────────────────────
    tick_diffs  = np.array(tick_diffs)
    median_diff = np.median(tick_diffs)
    actual_hz   = 1000.0 / median_diff if median_diff > 0 else 0
    print(f"  Median tick diff : {median_diff:.1f}  →  {actual_hz:.1f} Hz")
    print(f"  Messages         : {len(df)}   Duration: {df['time_s'].iloc[-1]:.2f} s")
    if   abs(actual_hz - 500) < 10: print("  ✓ 500 Hz confirmed")
    elif abs(actual_hz - 100) < 10: print("  ✓ 100 Hz confirmed")
    else:                           print(f"  ⚠ Unusual rate: {actual_hz:.1f} Hz")

    # ── Save full CSV ─────────────────────────────────────────────────────────
    csv_path = os.path.join(output_csv_dir, f"{base_name}_full.csv")
    df.to_csv(csv_path, index=False)
    print(f"  ✓ CSV → {csv_path}")

    # ============================================================
    # ── BUILD RAW ARRAYS (Go2 order) ─────────────────────────────
    # ============================================================
    q_raw  = df[[f"q_{leg}_{j}"   for leg in LEGS for j in ["hip","thigh","calf"]]].to_numpy()  # (N,12)
    qd_raw = df[[f"dq_{leg}_{j}"  for leg in LEGS for j in ["hip","thigh","calf"]]].to_numpy()  # (N,12)
    p_raw  = df[[f"ee_pos_{leg}_{ax}" for leg in LEGS for ax in "xyz"]].to_numpy()              # (N,12)
    v_raw  = df[[f"ee_vel_{leg}_{ax}" for leg in LEGS for ax in "xyz"]].to_numpy()              # (N,12)
    acc    = df[["acc_x",  "acc_y",  "acc_z" ]].to_numpy()                                      # (N,3)
    omega  = df[["gyro_x", "gyro_y", "gyro_z"]].to_numpy()                                      # (N,3)
    contacts_raw = df[["contact_FR","contact_FL","contact_RR","contact_RL"]].to_numpy().astype(np.int32)  # (N,4)

    # ── Reorder to deep-contact-estimator convention ──────────────────────────
    # [RH,RF,LH,LF] = [RR,FR,RL,FL] → JOINT_REORDER = [6,7,8,0,1,2,9,10,11,3,4,5]
    q = q_raw[:, JOINT_REORDER]
    qd= qd_raw[:,JOINT_REORDER]
    p = p_raw[:, JOINT_REORDER]
    v = v_raw[:, JOINT_REORDER]

    # Contact reorder: Go2[FR,FL,RR,RL] → model[RH,RF,LH,LF]=[RR,FR,RL,FL] → cols[2,0,3,1]
    contacts_reordered = contacts_raw[:, [2, 0, 3, 1]]
    decimal_labels     = np.dot(contacts_reordered, [8, 4, 2, 1]).astype(np.int32)

    # ── Trim all arrays to same length (safety) ───────────────────────────────
    arrays = [q, qd, p, v, acc, omega, contacts_raw, contacts_reordered, decimal_labels]
    min_len = min(a.shape[0] for a in arrays)
    if min_len < len(df):
        print(f"  ⚠ Trimming to min_length = {min_len}")
    q                  = q[:min_len]
    qd                 = qd[:min_len]
    p                  = p[:min_len]
    v                  = v[:min_len]
    acc                = acc[:min_len]
    omega              = omega[:min_len]
    contacts_raw       = contacts_raw[:min_len]
    contacts_reordered = contacts_reordered[:min_len]
    decimal_labels     = decimal_labels[:min_len]

    # ── Frame check ───────────────────────────────────────────────────────────
    # FK outputs are already in body frame by construction — this just confirms
    print("\n  Frame check (body frame: max abs < 0.6 m for EE positions):")
    check_frame("p (ee_pos)", p)
    check_frame("v (ee_vel)", v)
    check_frame("q (joints)", q)
    check_frame("acc",        acc)
    check_frame("omega",      omega)

    # ── Concatenate feature matrix ────────────────────────────────────────────
    preprocessed = np.concatenate([q, qd, p, v], axis=1)  # (N, 48)#acc, omega

    # ── Shape + label summary ─────────────────────────────────────────────────
    print(f"\n  Array shapes (all trimmed to N={min_len}):")
    print(f"  q            {q.shape}   joint pos  (model order)")
    print(f"  qd           {qd.shape}   joint vel  (model order)")
    print(f"  p            {p.shape}   EE pos     (body frame, model order)")
    print(f"  v            {v.shape}   EE vel     (body frame, model order)")
    print(f"  acc          {acc.shape}")
    print(f"  omega        {omega.shape}")
    print(f"  contacts     {contacts_raw.shape}   binary [FR,FL,RR,RL]")
    print(f"  dec_labels   {decimal_labels.shape}   0–15")
    print(f"  preprocessed {preprocessed.shape}   full feature matrix")

    print(f"\n  Label distribution:")
    unique, counts = np.unique(decimal_labels, return_counts=True)
    for u, c in zip(unique, counts):
        bar = "█" * int(c / min_len * 40)
        print(f"    label {u:2d}: {c:5d}  ({c/min_len*100:4.1f}%)  {bar}")

    # ── Save npy ──────────────────────────────────────────────────────────────
    npy_dir = os.path.join(output_npy_dir, base_name)
    os.makedirs(npy_dir, exist_ok=True)

    for name, arr in [
        ("q",                  q),
        ("qd",                 qd),
        ("p",                  p),
        ("v",                  v),
        ("acc",                acc),
        ("omega",              omega),
        ("contacts",           contacts_raw),
        ("contacts_reordered", contacts_reordered),
        ("contacts_labels",    decimal_labels),
        ("preprocessed_data",  preprocessed),
    ]:
        path = os.path.join(npy_dir, f"{name}.npy")
        np.save(path, arr)
        print(f"  ✓ {name}.npy  {arr.shape}  dtype={arr.dtype}")

    print(f"\n✓ Done → {npy_dir}")

In [ ]:
import numpy as np
import os

# paths
stiff_path = "./data/dataset_path"
soft_path  = "./data/dataset_path"

# load
X_stiff = np.load(os.path.join(stiff_path, "preprocessed_data.npy"))
y_stiff = np.load(os.path.join(stiff_path, "contacts_labels.npy"))

X_soft  = np.load(os.path.join(soft_path, "preprocessed_data.npy"))
y_soft  = np.load(os.path.join(soft_path, "contacts_labels.npy"))

print("Stiff:", X_stiff.shape, y_stiff.shape)
print("Soft :", X_soft.shape,  y_soft.shape)

# concatenate
X_combined = np.concatenate([X_stiff, X_soft], axis=0)
y_combined = np.concatenate([y_stiff, y_soft], axis=0)

print("Combined:", X_combined.shape, y_combined.shape)

# save
save_dir = "./data/dataset_path"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "preprocessed_data.npy"), X_combined)
np.save(os.path.join(save_dir, "contacts_labels.npy"), y_combined)

print("Saved combined dataset")